# Lab 10: Evaluation of Neural Machine Translation using BLEU Score

## Objective
- Evaluate a trained English→French Neural Machine Translation model.
- Compute BLEU-1 and BLEU-2 scores.
- Compare predicted translations with reference translations.
- Analyze translation quality.



# Theory

BLEU (Bilingual Evaluation Understudy) is a standard metric for evaluating machine translation systems.

- **BLEU-1** measures unigram (single-word) precision.
- **BLEU-2** measures bigram (two-word sequence) precision.

Higher BLEU values indicate closer agreement between the generated translation and the reference translation.

## Algorithm

- Load the trained model.
- Translate English sentences.
- Compare generated translations with reference translations.
- Compute BLEU-1.
- Compute BLEU-2.
- Calculate average BLEU scores.
- Display the results.


In [1]:
!pip -q install nltk


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


**Requirements**

In [4]:
from __future__ import unicode_literals, print_function, division
from io import open
import unicodedata
import re
import random

import torch
import torch.nn as nn
from torch import optim
import torch.nn.functional as F

import numpy as np
from torch.utils.data import TensorDataset, DataLoader, RandomSampler

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.set_default_device(device)
print(f"Using device = {torch.get_default_device()}")

Using device = cpu


## Loading Data

- To make a word list from sentence, let's create a Class the sentence/line and creates `word2index`, `index2word`, and `word2count`.

In [5]:
SOS_token = 0 # Start of the Sentence
EOS_token = 1 # End of the Sentence

class Lang:
    def __init__(self, name):
        self.name = name
        self.word2index = {}
        self.word2count = {}
        self.index2word = {0: "SOS", 1: "EOS"}
        self.n_words = 2  # Count SOS and EOS

    def addSentence(self, sentence):
        for word in sentence.split(' '):
            self.addWord(word)

    def addWord(self, word):
        if word not in self.word2index:
            self.word2index[word] = self.n_words
            self.word2count[word] = 1
            self.index2word[self.n_words] = word
            self.n_words += 1
        else:
            self.word2count[word] += 1

- The input `addSentence` method that is required to make the `word2index`, `index2word`, and `word2count` is (obviously) a sentence. 
- Therefore, so let's create a method to 
    - read the Dataset/file, 
    - split it into lines and then 
    - create sentence pairs (Language1 Sentence, Equivalent Language2 Sentence) 
    - Normalize.

In [6]:
# Turn a Unicode string to plain ASCII, thanks to
# https://stackoverflow.com/a/518232/2809427
def unicodeToAscii(s):
    return ''.join(
        c for c in unicodedata.normalize('NFD', s)
        if unicodedata.category(c) != 'Mn'
    )

# Lowercase, trim, and remove non-letter characters
def normalizeString(s):
    s = unicodeToAscii(s.lower().strip())
    s = re.sub(r"([.!?])", r" \1", s)
    s = re.sub(r"[^a-zA-Z!?]+", r" ", s)
    return s.strip()

In [7]:
def readLangs(path:str):
    lang1 = 'eng'; lang2 = 'fra'
    print("Reading lines...")

    # Read the file and split into lines
    lines = open(path, encoding='utf-8').\
        read().strip().split('\n')

    # Split every line into pairs and normalize (english to french)
    pairs = [[normalizeString(s) for s in l.split('\t')] for l in lines]

    # Reverse pairs: English-French -> French-English
    pairs = [list(reversed(p)) for p in pairs]

    # Input is French, output is English
    input_lang = Lang(lang2)
    output_lang = Lang(lang1)

    return input_lang, output_lang, pairs

Since there are a lot of example sentences and we want to train something quickly, we’ll trim the data set to only relatively short and simple sentences. Here the maximum length is 10 words (that includes ending punctuation) and we’re filtering to sentences that translate to the form “I am” or “He is” etc. (accounting for apostrophes replaced earlier).

The overall purpose is to clean and prepare sentence pairs for training a language model by removing unsuitable examples.

In [8]:
MAX_LENGTH = 5

eng_prefixes = (
    "i am ", "i m ",
    "he is", "he s ",
    "she is", "she s ",
    "you are", "you re ",
    "we are", "we re ",
    "they are", "they re "
)

def filterPair(p):
    return len(p[0].split(' ')) < MAX_LENGTH and \
        len(p[1].split(' ')) < MAX_LENGTH and \
        p[1].startswith(eng_prefixes)


def filterPairs(pairs):
    return [pair for pair in pairs if filterPair(pair)]

In [9]:
def prepareData(path):
    input_lang, output_lang, pairs = readLangs(path)
    print("Read %s sentence pairs" % len(pairs))
    pairs = filterPairs(pairs)
    print("Trimmed to %s sentence pairs" % len(pairs))
    print("Counting words...")
    for pair in pairs:
        input_lang.addSentence(pair[0])
        output_lang.addSentence(pair[1])
    print("Counted words:")
    print(input_lang.name, input_lang.n_words)
    print(output_lang.name, output_lang.n_words)
    return input_lang, output_lang, pairs

In [10]:
PATH = 'data/eng-fra.txt'

input_lang, output_lang, pairs = prepareData(PATH)
print(random.choice(pairs))

output_lang.word2index['am']  # try different English words. 

Reading lines...
Read 135842 sentence pairs
Trimmed to 3272 sentence pairs
Counting words...
Counted words:
fra 1757
eng 967
['je suis desolee', 'i am sorry']


15

### Encoder

In [11]:
class EncoderRNN(nn.Module):
    def __init__(self, input_size, hidden_size, dropout_p=0.1):
        super(EncoderRNN, self).__init__()
        self.hidden_size = hidden_size

        self.embedding = nn.Embedding(input_size, hidden_size)
        self.rnn = nn.RNN(hidden_size, hidden_size, batch_first=True)
        self.dropout = nn.Dropout(dropout_p)

    def forward(self, input):
        embedded = self.dropout(self.embedding(input))
        output, hidden = self.rnn(embedded)
        return output, hidden

### Decoder

In [12]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class LuongDotAttention(nn.Module):
    def __init__(self, hidden_size):
        super(LuongDotAttention, self).__init__()

        # For:
        # s~_t = tanh(W_c[c_t;s_t])
        self.Wc = nn.Linear(hidden_size * 2, hidden_size)


    def forward(self, query, keys):
        """
        query:
            Current decoder hidden state s_t
            Shape: (batch_size, 1, hidden_size)

        keys:
            Encoder hidden states h_1,...,h_T
            Shape: (batch_size, seq_len, hidden_size)

        Returns:
            attentional_hidden:
                s~_t
                Shape: (batch_size, 1, hidden_size)

            weights:
                attention weights alpha_t
                Shape: (batch_size, 1, seq_len)
        """

        # Alignment scores:
        # e_{t,i} = s_t^T h_i
        scores = torch.bmm(
            query,
            keys.transpose(1, 2)
        )

        # Attention weights:
        # alpha_{t,i} = softmax(e_{t,i})
        weights = F.softmax(scores, dim=-1)

        # Context vector:
        # c_t = sum(alpha_{t,i} * h_i)
        context = torch.bmm(
            weights,
            keys
        )

        # Concatenate context and decoder hidden state:
        # [c_t ; s_t]
        combined = torch.cat(
            (context, query),
            dim=-1
        )

        # Attentional hidden state:
        # s~_t = tanh(W_c[c_t;s_t])
        attentional_hidden = torch.tanh(
            self.Wc(combined)
        )

        return attentional_hidden, weights

In [13]:
class AttnDecoderRNN(nn.Module):
    def __init__(self, hidden_size, output_size, dropout_p=0.1):
        super(AttnDecoderRNN, self).__init__()
        self.embedding = nn.Embedding(output_size, hidden_size)
        self.attention = LuongDotAttention(hidden_size)
        self.rnn = nn.RNN(2 * hidden_size, hidden_size, batch_first=True)
        self.out = nn.Linear(hidden_size, output_size)
        self.dropout = nn.Dropout(dropout_p)

    def forward(self, encoder_outputs, encoder_hidden, target_tensor=None):
        batch_size = encoder_outputs.size(0)
        decoder_input = torch.empty(batch_size, 1, dtype=torch.long, device=device).fill_(SOS_token)
        decoder_hidden = encoder_hidden
        decoder_outputs = []
        attentions = []

        for i in range(MAX_LENGTH):
            decoder_output, decoder_hidden, attn_weights = self.forward_step(
                decoder_input, decoder_hidden, encoder_outputs
            )
            decoder_outputs.append(decoder_output)
            attentions.append(attn_weights)

            if target_tensor is not None:
                # Teacher forcing: Feed the target as the next input
                decoder_input = target_tensor[:, i].unsqueeze(1) # Teacher forcing
            else:
                # Without teacher forcing: use its own predictions as the next input
                _, topi = decoder_output.topk(1)
                decoder_input = topi.squeeze(-1).detach()  # detach from history as input

        decoder_outputs = torch.cat(decoder_outputs, dim=1)
        decoder_outputs = F.log_softmax(decoder_outputs, dim=-1)
        attentions = torch.cat(attentions, dim=1)

        return decoder_outputs, decoder_hidden, attentions


    def forward_step(self, input, hidden, encoder_outputs):
        embedded =  self.dropout(self.embedding(input))

        query = hidden.permute(1, 0, 2) # seq_len, batch, hidden_size -> batch, seq_len, hidden_size
        context, attn_weights = self.attention(query, encoder_outputs)
        input_rnn = torch.cat((embedded, context), dim=2)

        output, hidden = self.rnn(input_rnn, hidden)
        output = self.out(output)

        return output, hidden, attn_weights

## Training



In [14]:
def indexesFromSentence(lang, sentence):
    return [lang.word2index[word] for word in sentence.split(' ')]

def tensorFromSentence(lang, sentence):
    indexes = indexesFromSentence(lang, sentence)
    indexes.append(EOS_token)
    return torch.tensor(indexes, dtype=torch.long, device=device).view(1, -1)

def tensorsFromPair(pair):
    input_tensor = tensorFromSentence(input_lang, pair[0])
    target_tensor = tensorFromSentence(output_lang, pair[1])
    return (input_tensor, target_tensor)

def get_dataloader(batch_size):
    input_lang, output_lang, pairs = prepareData(path=PATH)

    n = len(pairs)
    input_ids = np.zeros((n, MAX_LENGTH), dtype=np.int32)
    target_ids = np.zeros((n, MAX_LENGTH), dtype=np.int32)

    for idx, (inp, tgt) in enumerate(pairs):
        inp_ids = indexesFromSentence(input_lang, inp)
        tgt_ids = indexesFromSentence(output_lang, tgt)
        inp_ids.append(EOS_token)
        tgt_ids.append(EOS_token)
        input_ids[idx, :len(inp_ids)] = inp_ids
        target_ids[idx, :len(tgt_ids)] = tgt_ids

    train_data = TensorDataset(torch.LongTensor(input_ids).to(device),
                               torch.LongTensor(target_ids).to(device))

    train_sampler = RandomSampler(train_data)
    train_dataloader = DataLoader(train_data, sampler=train_sampler, batch_size=batch_size)
    return input_lang, output_lang, train_dataloader

### Training Loop

In [15]:
def train_epoch(dataloader, encoder, decoder, encoder_optimizer,
          decoder_optimizer, criterion):

    total_loss = 0
    for data in dataloader:
        input_tensor, target_tensor = data

        encoder_optimizer.zero_grad()
        decoder_optimizer.zero_grad()

        encoder_outputs, encoder_hidden = encoder(input_tensor)
        decoder_outputs, _, _ = decoder(encoder_outputs, encoder_hidden, target_tensor) # using teacher forcing

        loss = criterion(
            decoder_outputs.view(-1, decoder_outputs.size(-1)),
            target_tensor.view(-1)
        )
        loss.backward()

        encoder_optimizer.step()
        decoder_optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)

In [16]:
import time
import math

def asMinutes(s):
    m = math.floor(s / 60)
    s -= m * 60
    return '%dm %ds' % (m, s)

def timeSince(since, percent):
    now = time.time()
    s = now - since
    es = s / (percent)
    rs = es - s
    return '%s (- %s)' % (asMinutes(s), asMinutes(rs))

In [17]:
import matplotlib.pyplot as plt
plt.switch_backend('agg')
import matplotlib.ticker as ticker
import numpy as np

def showPlot(points):
    plt.figure()
    fig, ax = plt.subplots()
    # this locator puts ticks at regular intervals
    loc = ticker.MultipleLocator(base=0.2)
    ax.yaxis.set_major_locator(loc)
    plt.plot(points)

In [18]:
def train(train_dataloader, encoder, decoder, n_epochs, learning_rate=0.001,
               print_every=100, plot_every=100):
    start = time.time()
    plot_losses = []
    print_loss_total = 0  # Reset every print_every
    plot_loss_total = 0  # Reset every plot_every

    encoder_optimizer = optim.Adam(encoder.parameters(), lr=learning_rate)
    decoder_optimizer = optim.Adam(decoder.parameters(), lr=learning_rate)
    criterion = nn.NLLLoss()

    for epoch in range(1, n_epochs + 1):
        loss = train_epoch(train_dataloader, encoder, decoder, encoder_optimizer, decoder_optimizer, criterion)
        print_loss_total += loss
        plot_loss_total += loss

        if epoch % print_every == 0:
            print_loss_avg = print_loss_total / print_every
            print_loss_total = 0
            print('%s (%d %d%%) %.4f' % (timeSince(start, epoch / n_epochs),
                                        epoch, epoch / n_epochs * 100, print_loss_avg))

        if epoch % plot_every == 0:
            plot_loss_avg = plot_loss_total / plot_every
            plot_losses.append(plot_loss_avg)
            plot_loss_total = 0

    showPlot(plot_losses)

## Evaluation Code

In [19]:
def evaluate(encoder, decoder, sentence, input_lang, output_lang):
    with torch.no_grad():
        input_tensor = tensorFromSentence(input_lang, sentence)

        encoder_outputs, encoder_hidden = encoder(input_tensor)
        decoder_outputs, decoder_hidden, decoder_attn = decoder(encoder_outputs, encoder_hidden)

        _, topi = decoder_outputs.topk(1)
        decoded_ids = topi.squeeze()

        decoded_words = []
        for idx in decoded_ids:
            if idx.item() == EOS_token:
                decoded_words.append('<EOS>')
                break
            decoded_words.append(output_lang.index2word[idx.item()])
    return decoded_words, decoder_attn

In [20]:
def evaluateRandomly(encoder, decoder, n=10):
    for i in range(n):
        pair = random.choice(pairs)
        print('>', pair[0])
        print('=', pair[1])
        output_words, _ = evaluate(encoder, decoder, pair[0], input_lang, output_lang)
        output_sentence = ' '.join(output_words)
        print('<', output_sentence)
        print('')

### Training and Evaluating

In [28]:
hidden_size = 128
batch_size = 32
EPOCHS = 1000

input_lang, output_lang, train_dataloader = get_dataloader(batch_size)

encoder = EncoderRNN(input_lang.n_words, hidden_size).to(device)
decoder = AttnDecoderRNN(hidden_size, output_lang.n_words).to(device)

train(train_dataloader, encoder, decoder, EPOCHS, print_every=5, plot_every=5)

Reading lines...
Read 135842 sentence pairs
Trimmed to 3272 sentence pairs
Counting words...
Counted words:
fra 1757
eng 967
0m 3s (- 12m 9s) (5 0%) 1.8900
0m 7s (- 12m 6s) (10 1%) 1.2073
0m 11s (- 12m 21s) (15 1%) 0.9538
0m 15s (- 12m 31s) (20 2%) 0.7690
0m 19s (- 12m 32s) (25 2%) 0.6143
0m 22s (- 12m 15s) (30 3%) 0.4877
0m 26s (- 12m 20s) (35 3%) 0.3850
0m 30s (- 12m 18s) (40 4%) 0.3213
0m 34s (- 12m 12s) (45 4%) 0.2721
0m 38s (- 12m 2s) (50 5%) 0.2437
0m 41s (- 11m 54s) (55 5%) 0.2078
0m 45s (- 11m 45s) (60 6%) 0.1911
0m 48s (- 11m 38s) (65 6%) 0.1718
0m 52s (- 11m 36s) (70 7%) 0.1673
0m 56s (- 11m 33s) (75 7%) 0.1523
0m 59s (- 11m 29s) (80 8%) 0.1507
1m 3s (- 11m 26s) (85 8%) 0.1413
1m 7s (- 11m 21s) (90 9%) 0.1352
1m 11s (- 11m 18s) (95 9%) 0.1351
1m 14s (- 11m 13s) (100 10%) 0.1219
1m 18s (- 11m 8s) (105 10%) 0.1172
1m 21s (- 11m 3s) (110 11%) 0.1140
1m 25s (- 10m 58s) (115 11%) 0.1131
1m 29s (- 10m 52s) (120 12%) 0.1163
1m 32s (- 10m 50s) (125 12%) 0.1068
1m 36s (- 10m 45s) (130

In [29]:
encoder.eval()
decoder.eval()
evaluateRandomly(encoder, decoder)

> il est extremement heureux
= he s extremely happy
< he s extremely happy <EOS>

> vous etes tres genereuses
= you re very generous
< you re very generous <EOS>

> nous sommes sournois
= we re sneaky
< we re sneaky <EOS>

> je suis tres heureuse
= i m very happy
< i m very happy <EOS>

> je suis de kyoto
= i m from kyoto
< i m from kyoto <EOS>

> je suis coincee
= i m stuck
< i m stuck <EOS>

> elle est affutee
= she is sharp
< you re all infected <EOS>

> il est detective
= he is a detective
< we re here early <EOS>

> je suis professeur
= i m a teacher
< we re bored <EOS>

> vous etes trop bruyantes
= you re too loud
< you re too loud <EOS>



In [30]:
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

smooth = SmoothingFunction().method1


In [31]:
def calculate_bleu(encoder,
                   decoder,
                   input_lang,
                   output_lang,
                   pairs,
                   samples=100):

    bleu1_scores=[]
    bleu2_scores=[]

    sample_pairs = random.sample(pairs, min(samples, len(pairs)))

    for pair in sample_pairs:

        source=pair[0]
        target=pair[1]

        output_words,_=evaluate(
            encoder,
            decoder,
            source,
            input_lang,
            output_lang
        )

        if "<EOS>" in output_words:
            output_words.remove("<EOS>")

        reference=[target.split()]
        candidate=output_words

        bleu1=sentence_bleu(
            reference,
            candidate,
            weights=(1,0,0,0),
            smoothing_function=smooth
        )

        bleu2=sentence_bleu(
            reference,
            candidate,
            weights=(0.5,0.5,0,0),
            smoothing_function=smooth
        )

        bleu1_scores.append(bleu1)
        bleu2_scores.append(bleu2)

        print("="*60)
        print("French Input        :", source)
        print("Reference (English) :", target)
        print("Prediction (English):", " ".join(candidate))
        print("BLEU-1    :",round(bleu1,4))
        print("BLEU-2    :",round(bleu2,4))

    print("\nAverage BLEU-1:",sum(bleu1_scores)/len(bleu1_scores))
    print("Average BLEU-2:",sum(bleu2_scores)/len(bleu2_scores))


In [33]:
calculate_bleu(
    encoder,
    decoder,
    input_lang,
    output_lang,
    pairs,
    samples=100
)

French Input        : je suis gave
Reference (English) : i m stuffed
Prediction (English): i m a salesperson
BLEU-1    : 0.5
BLEU-2    : 0.4082
French Input        : nous sommes anxieux
Reference (English) : we re anxious
Prediction (English): we re anxious
BLEU-1    : 1.0
BLEU-2    : 1.0
French Input        : il est americain
Reference (English) : he is american
Prediction (English): we re beautiful
BLEU-1    : 0
BLEU-2    : 0
French Input        : je suis paresseuse
Reference (English) : i m lazy
Prediction (English): we re bored
BLEU-1    : 0
BLEU-2    : 0
French Input        : je suis connue
Reference (English) : i m famous
Prediction (English): we re bored
BLEU-1    : 0
BLEU-2    : 0
French Input        : vous etes connues
Reference (English) : you re famous
Prediction (English): you re famous
BLEU-1    : 1.0
BLEU-2    : 1.0
French Input        : je viens de roumanie
Reference (English) : i m from romania
Prediction (English): i m from romania
BLEU-1    : 1.0
BLEU-2    : 1.0
Frenc

# Discussion

The Neural Machine Translation model incorporating the Luong Attention mechanism was effectively trained to translate French sentences into English. By enabling the decoder to concentrate on the most pertinent parts of the input sequence, the attention mechanism enhanced translation accuracy. Performance was assessed using BLEU-1 and BLEU-2 metrics, which evaluate the overlap of individual words and word pairs between generated and reference translations. While certain sentences received perfect scores, others scored lower due to variations in vocabulary or syntactic structure. Overall BLEU results suggested that the model captured meaningful translation patterns but still struggled with more complex constructions. Translation performance could potentially be improved by increasing training duration, expanding the dataset, or refining hyperparameters.






# Conclusion

This lab involved the implementation and evaluation of a Seq2Seq neural machine translation model incorporating the Luong Attention mechanism. The system was designed to translate French sentences into English, showing how attention improves translation accuracy by focusing on relevant parts of the input sequence. Performance was measured using BLEU-1 and BLEU-2 scores, offering an objective assessment of output quality. While not all translations were flawless, the model delivered acceptable results across a number of test cases. The exercise illustrated the integration of attention-based architectures with standard evaluation metrics in developing and analyzing machine translation models.